# 03 — Model Comparison Experiments (Phase ML-3/ML-4)

Loads `train.pkl` / `test.pkl` from `02_preprocessing_and_feature_engineering.ipynb`.

Compares **6 algorithms** (Logistic Regression, Decision Tree, Random Forest,
XGBoost, SVM, KNN) across **3 feature sets** (A=Basic, B=Basic+Enhanced,
C=Extended/Advanced), using leakage-safe pipelines, stratified 5-fold CV on
the 80% training split, and a single final evaluation of the selected
winners on the untouched 20% test split.

**Algorithm-list conflict, reported as instructed rather than silently
resolved:** this project's `CLAUDE.md` (§14) mandates exactly Logistic
Regression, Decision Tree, SVM, and Random Forest. The newer ML-integration
brief asks for Logistic Regression, Random Forest, XGBoost, SVM, and KNN.
**Decision Tree is included here as an additional baseline** (not silently
dropped) so both requirements are satisfied — see the results below for how
it performs relative to the other five.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, brier_score_loss, make_scorer)
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET = 'Heart Disease'
pd.set_option('display.width', 160)

train_df = pd.read_pickle("train.pkl")
test_df = pd.read_pickle("test.pkl")
print("Train:", train_df.shape, "Test:", test_df.shape)

Train: (828, 30) Test: (207, 30)


## 1. Feature sets (from notebook 02) and leakage-safe pipeline builder

In [2]:
BASIC_NUMERIC = ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)']
BASIC_BINARY = ['Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain']
BASIC_CATEGORICAL = ['Sex']
ENHANCED_NUMERIC_ADD = ['Total_Cholesterol(mg/dL)', 'HDL(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'MaxHR']
ADVANCED_NUMERIC_ADD = ['Troponin_I_harmonised', 'Sodium(mmol/L)', 'Potassium', 'Chloride', 'Creatinine(mg/dL)', 'Platelets', 'Himoglobin']
ADVANCED_BINARY_ADD = ['Troponin_Censored']

FEATURE_SETS = {
    'A_Basic': {'numeric': BASIC_NUMERIC, 'binary': BASIC_BINARY, 'categorical': BASIC_CATEGORICAL},
    'B_Basic_Enhanced': {'numeric': BASIC_NUMERIC + ENHANCED_NUMERIC_ADD, 'binary': BASIC_BINARY, 'categorical': BASIC_CATEGORICAL},
    'C_Extended_Advanced': {'numeric': BASIC_NUMERIC + ENHANCED_NUMERIC_ADD + ADVANCED_NUMERIC_ADD,
                             'binary': BASIC_BINARY + ADVANCED_BINARY_ADD, 'categorical': BASIC_CATEGORICAL},
}

def make_pipeline(feature_set, model):
    # Every preprocessing step (imputation incl. missingness indicators, scaling,
    # one-hot encoding) is fit ONLY within each CV fold's training portion via
    # this Pipeline+ColumnTransformer - never on the full dataset before splitting.
    numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())])
    binary_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent'))])
    cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
    pre = ColumnTransformer([('num', numeric_pipe, feature_set['numeric']), ('bin', binary_pipe, feature_set['binary']), ('cat', cat_pipe, feature_set['categorical'])])
    return Pipeline([('preprocess', pre), ('model', model)])

## 2. Algorithms and hyperparameter grids

In [3]:
neg, pos = (train_df[TARGET]==0).sum(), (train_df[TARGET]==1).sum()
scale_pos_weight = neg / pos
print(f"Class imbalance -> scale_pos_weight for XGBoost = {scale_pos_weight:.3f}")

ALGORITHMS = {
    'LogisticRegression': (LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
                            {'model__C': [0.01, 0.1, 1, 10]}),
    'DecisionTree': (DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
                      {'model__max_depth': [3, 5, 8, None], 'model__min_samples_leaf': [1, 5, 10]}),
    'RandomForest': (RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
                      {'model__n_estimators': [200, 400], 'model__max_depth': [6, 10, None], 'model__max_features': ['sqrt', 'log2']}),
    'XGBoost': (XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=scale_pos_weight, n_jobs=-1),
                {'model__n_estimators': [200, 400], 'model__max_depth': [3, 5], 'model__learning_rate': [0.05, 0.1]}),
    'SVM': (SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE),
            {'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 'auto']}),
    'KNN': (KNeighborsClassifier(), {'model__n_neighbors': [5, 11, 15, 21], 'model__weights': ['uniform', 'distance']}),
}

Class imbalance -> scale_pos_weight for XGBoost = 0.769


## 3. Stratified 5-fold CV grid search: all 6 algorithms x all 3 feature sets

Model selection uses only the 80% training split (`train_df`) - the 20% test
split is never touched in this cell.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_train, y_train = train_df, train_df[TARGET]

all_results = []
best_estimators = {}
t0 = time.time()
for fs_name, fs in FEATURE_SETS.items():
    for algo_name, (model, grid) in ALGORITHMS.items():
        pipe = make_pipeline(fs, model)
        gs = GridSearchCV(pipe, grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
        gs.fit(X_train, y_train)
        idx = gs.best_index_
        all_results.append({'feature_set': fs_name, 'algorithm': algo_name, 'best_params': gs.best_params_,
                             'cv_mean_auc': gs.cv_results_['mean_test_score'][idx], 'cv_sd_auc': gs.cv_results_['std_test_score'][idx]})
        best_estimators[(fs_name, algo_name)] = gs.best_estimator_
        print(f"[{time.time()-t0:6.1f}s] {fs_name:22s} {algo_name:20s} CV-AUC={all_results[-1]['cv_mean_auc']:.4f}+/-{all_results[-1]['cv_sd_auc']:.4f}")

results_df = pd.DataFrame(all_results)
print(f"\nTotal wall time: {time.time()-t0:.1f}s")

[  11.3s] A_Basic                LogisticRegression   CV-AUC=0.9040+/-0.0263


[  13.4s] A_Basic                DecisionTree         CV-AUC=0.8703+/-0.0217


[  60.3s] A_Basic                RandomForest         CV-AUC=0.9281+/-0.0252


[  66.8s] A_Basic                XGBoost              CV-AUC=0.9298+/-0.0233


[  70.9s] A_Basic                SVM                  CV-AUC=0.9224+/-0.0235


...

[ 136.4s] C_Extended_Advanced    DecisionTree         CV-AUC=0.9599+/-0.0166


[ 182.8s] C_Extended_Advanced    RandomForest         CV-AUC=0.9986+/-0.0011


[ 190.0s] C_Extended_Advanced    XGBoost              CV-AUC=0.9969+/-0.0018


[ 193.2s] C_Extended_Advanced    SVM                  CV-AUC=0.9877+/-0.0049


[ 194.6s] C_Extended_Advanced    KNN                  CV-AUC=0.9775+/-0.0029

Total wall time: 194.7s


In [5]:
pivot = results_df.pivot(index='algorithm', columns='feature_set', values='cv_mean_auc')
pivot = pivot[['A_Basic', 'B_Basic_Enhanced', 'C_Extended_Advanced']]
pivot.round(4)

feature_set,A_Basic,B_Basic_Enhanced,C_Extended_Advanced
algorithm,,,
DecisionTree,0.8703,0.9302,0.9599
KNN,0.9081,0.9705,0.9775
LogisticRegression,0.9040,0.9784,0.9866
RandomForest,0.9281,0.9819,0.9986
SVM,0.9224,0.9810,0.9877
XGBoost,0.9298,0.9817,0.9969


**Decision Tree consistently ranks weakest of the 6 algorithms in every
feature set** - included as required by `CLAUDE.md`, but not competitive
against the ensemble/kernel methods on this dataset. This resolves the
algorithm-list conflict by testing fairly rather than assuming either list
was right.

**The A -> B -> C progression shows a large, real jump in CV-AUC** as more
clinical/lab information is added - this is the core research question's
first piece of evidence (see the written report for the full discussion of
why the C-group's near-ceiling AUC is treated with caution, not celebrated
uncritically - it echoes the same abnormally-high-correlation concern
documented in `dataset_audit.md`).

## 4. Full metric suite for the top contenders in each feature set (not AUC alone)

In [6]:
def specificity_score(y_true, y_pred):
    tn = ((y_true==0)&(y_pred==0)).sum(); fp = ((y_true==0)&(y_pred==1)).sum()
    return tn/(tn+fp) if (tn+fp) else np.nan

scoring = {'roc_auc': 'roc_auc', 'accuracy': 'accuracy', 'precision': 'precision',
           'recall': 'recall', 'f1': 'f1', 'specificity': make_scorer(specificity_score)}

detailed = []
for fs_name in FEATURE_SETS:
    sub = results_df[results_df['feature_set']==fs_name].sort_values('cv_mean_auc', ascending=False)
    for algo_name in sub['algorithm'].head(4):
        pipe = best_estimators[(fs_name, algo_name)]
        cvres = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        row = {'feature_set': fs_name, 'algorithm': algo_name}
        for k in scoring:
            row[f'cv_{k}'] = cvres[f'test_{k}'].mean()
        detailed.append(row)

detailed_df = pd.DataFrame(detailed)
detailed_df.round(4)

,feature_set,algorithm,cv_roc_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_specificity
0,A_Basic,XGBoost,0.9298,0.8551,0.9102,0.8269,0.8657,0.8917
1,A_Basic,RandomForest,0.9281,0.8563,0.9075,0.8313,0.8672,0.8889
2,A_Basic,SVM,0.9224,0.8552,0.9179,0.8184,0.8648,0.9028
3,A_Basic,KNN,0.9081,0.8056,0.9101,0.7287,0.8091,0.9056
4,B_Basic_Enhanced,RandomForest,0.9819,0.9396,0.9647,0.9274,0.9456,0.9556
5,B_Basic_Enhanced,XGBoost,0.9817,0.9312,0.9479,0.9295,0.9385,0.9333
6,B_Basic_Enhanced,SVM,0.9810,0.9433,0.9774,0.9210,0.9483,0.9722
7,B_Basic_Enhanced,LogisticRegression,0.9784,0.9312,0.9562,0.9210,0.9380,0.9444
8,C_Extended_Advanced,RandomForest,0.9986,0.9770,0.9893,0.9701,0.9795,0.9861
9,C_Extended_Advanced,XGBoost,0.9969,0.9674,0.9869,0.9552,0.9707,0.9833


## 5. Final evaluation on the held-out 20% test set

Per leakage-safe methodology, the test set is used **exactly once per
feature-set winner** here - not for every one of the 18 trained
configurations (that would itself be a form of test-set overfitting via
selection). Random Forest and XGBoost were the closest contenders in CV for
every feature set, so both are evaluated for direct comparison; the actual
recommendation is made in the written report using this test evidence
together with the CV evidence above.

In [7]:
def specificity(y_true, y_pred):
    tn = ((y_true==0)&(y_pred==0)).sum(); fp = ((y_true==0)&(y_pred==1)).sum()
    return tn/(tn+fp)

def evaluate_on_test(fs_name, algo_name):
    pipe = best_estimators[(fs_name, algo_name)]
    proba = pipe.predict_proba(test_df)[:, 1]
    pred = pipe.predict(test_df)
    y = test_df[TARGET]
    return {
        'feature_set': fs_name, 'algorithm': algo_name,
        'test_roc_auc': roc_auc_score(y, proba), 'test_accuracy': accuracy_score(y, pred),
        'test_precision': precision_score(y, pred), 'test_recall': recall_score(y, pred),
        'test_specificity': specificity(y, pred), 'test_f1': f1_score(y, pred),
        'test_brier': brier_score_loss(y, proba),
    }

test_rows = []
for fs_name in FEATURE_SETS:
    for algo_name in ['RandomForest', 'XGBoost']:
        test_rows.append(evaluate_on_test(fs_name, algo_name))
test_results_df = pd.DataFrame(test_rows)
test_results_df.round(4)

,feature_set,algorithm,test_roc_auc,test_accuracy,test_precision,test_recall,test_specificity,test_f1,test_brier
0,A_Basic,RandomForest,0.9136,0.8309,0.8661,0.8291,0.8333,0.8472,0.1208
1,A_Basic,XGBoost,0.9253,0.8454,0.8899,0.8291,0.8667,0.8584,0.1128
2,B_Basic_Enhanced,RandomForest,0.9753,0.8986,0.9528,0.8632,0.9444,0.9058,0.0722
3,B_Basic_Enhanced,XGBoost,0.9802,0.9227,0.9469,0.9145,0.9333,0.9304,0.0594
4,C_Extended_Advanced,RandomForest,0.9919,0.9469,0.9649,0.9402,0.9556,0.9524,0.0415
5,C_Extended_Advanced,XGBoost,0.9887,0.9517,0.9735,0.9402,0.9667,0.9565,0.0385


In [8]:
majority = y_train.mode()[0]
baseline_acc = accuracy_score(test_df[TARGET], np.full(len(test_df), majority))
print(f"Majority-class baseline test accuracy: {baseline_acc:.4f} (every model above must clear this trivially, and does)")

Majority-class baseline test accuracy: 0.5652 (every model above must clear this trivially, and does)


## 6. Confusion matrices for the two deployment candidates

(Basic and Basic+Enhanced XGBoost - see the written report for the full
feature-trimming pass that follows in notebook 04, which finalises the
recommended Enhanced feature list.)

In [9]:
for fs_name in ['A_Basic', 'B_Basic_Enhanced']:
    pipe = best_estimators[(fs_name, 'XGBoost')]
    pred = pipe.predict(test_df)
    cm = confusion_matrix(test_df[TARGET], pred)
    print(f"{fs_name} / XGBoost confusion matrix [[TN FP][FN TP]]:\n{cm}\n")

A_Basic / XGBoost confusion matrix [[TN FP][FN TP]]:
[[78 12]
 [20 97]]



B_Basic_Enhanced / XGBoost confusion matrix [[TN FP][FN TP]]:
[[ 84   6]
 [ 10 107]]



In [10]:
import pickle
with open("best_estimators.pkl", "wb") as f:
    pickle.dump(best_estimators, f)
results_df.to_pickle("cv_results.pkl")
test_results_df.to_pickle("test_results.pkl")
print("Saved best_estimators.pkl, cv_results.pkl, test_results.pkl for notebook 04.")

Saved best_estimators.pkl, cv_results.pkl, test_results.pkl for notebook 04.
